In [2]:
# -*- coding: utf-8 -*-
import os
import re
import glob
import json
import numpy as np
import pandas as pd

import optuna
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score

# =========================
# 0) 配置
# =========================
RANDOM_SEED = 42
N_TRIALS = 30
N_SPLITS = 5
THRESH = 0.5

# 直接读取你已生成的特征文件（xlsx）
FEATURE_FILE_GLOB = "./Malodors_Rule&FG&StructKG_features.xlsx"
# 如果你的文件名不同，改成你的，例如：
# FEATURE_FILE_GLOB = "./Malodors_transformed_MORGAN_features.xlsx"

# 固定参数（Optuna 只搜索部分超参）
BASE_XGB_PARAMS = dict(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    n_jobs=-1,
    random_state=RANDOM_SEED,
    verbosity=0,
    multi_strategy="multi_output_tree",
)

N_PARALLEL = min(5, os.cpu_count() or 1)

# =========================
# 1) 版本检查
# =========================
def _ver_tuple(v: str):
    parts = re.split(r"[.+-]", v.strip())
    nums = []
    for p in parts[:3]:
        try:
            nums.append(int(p))
        except Exception:
            nums.append(0)
    while len(nums) < 3:
        nums.append(0)
    return tuple(nums)

def check_xgb_version():
    v = _ver_tuple(xgb.__version__)
    if v < (1, 6, 0):
        raise RuntimeError(f"xgboost=={xgb.__version__} 太旧：multi-label 需要 >=1.6")
    if "multi_strategy" in BASE_XGB_PARAMS and v < (2, 0, 0):
        raise RuntimeError(
            f"xgboost=={xgb.__version__} 不支持 multi_output_tree（需要>=2.0）。"
            f"升级或删掉 multi_strategy。"
        )

# =========================
# 2) 找文件 + 按列位置强制切分 X / y
# =========================
def find_feature_file(pattern: str):
    cands = sorted(glob.glob(pattern))
    if not cands:
        raise FileNotFoundError(
            f"找不到特征文件：{pattern}\n"
            f"请确认你保存的 xlsx 文件名，或修改 FEATURE_FILE_GLOB。"
        )
    return cands[-1]

def _to_binary01_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    将 y 的 138 列尽量转成 0/1：
    - bool -> 0/1
    - 数值 -> (>=0.5) 视为 1，否则 0
    - 字符串 -> 识别 {'1','0','true','false','yes','no'} 等
    """
    out = df.copy()

    for c in out.columns:
        s = out[c]

        if s.dtype == bool:
            out[c] = s.astype(int)
            continue

        if np.issubdtype(s.dtype, np.number):
            out[c] = (s.fillna(0).astype(float) >= 0.5).astype(int)
            continue

        # object / string
        ss = s.astype(str).str.strip().str.lower()
        true_set = {"1", "true", "t", "yes", "y"}
        false_set = {"0", "false", "f", "no", "n", "nan", "none", ""}

        out[c] = ss.map(lambda x: 1 if x in true_set else (0 if x in false_set else np.nan))
        if out[c].isna().any():
            # 有些值不是标准 true/false/0/1，就尽量转数值再二值化
            coerced = pd.to_numeric(s, errors="coerce")
            if coerced.notna().any():
                out[c] = (coerced.fillna(0).astype(float) >= 0.5).astype(int)
            else:
                # 实在无法解析，缺失视为0
                out[c] = out[c].fillna(0).astype(int)

    return out

def load_Xy_by_fixed_columns(xlsx_path: str, n_labels: int = 138):
    """
    你要求的固定切分：
      第 1 列(索引0)           -> SMILES（不进模型）
      第 2~第139列(索引1~138) -> y（共 138 列）
      第 140列起(索引>=139)   -> X（训练特征）

    返回：
      X, y, feature_cols, label_cols, smiles_col_name, df_all
    """
    df = pd.read_excel(xlsx_path)

    if df.shape[1] < 1 + n_labels + 1:
        raise ValueError(
            f"列数不够：当前 {df.shape[1]} 列，但至少需要 1(SMILES)+{n_labels}(y)+1(X) = {1+n_labels+1} 列。\n"
            f"请检查文件是否缺列/被截断。"
        )

    smiles_col = df.columns[0]
    label_cols = list(df.columns[1 : 1 + n_labels])
    feature_cols = list(df.columns[1 + n_labels : ])

    # y：强制二值化
    y_df = _to_binary01_df(df[label_cols])

    # X：数值化 + NaN填0
    X_df = df[feature_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).astype(np.float32)

    X = X_df.values
    y = y_df.fillna(0).astype(int).values

    # 一些健壮性检查
    if X.shape[0] != y.shape[0]:
        raise ValueError("X 和 y 行数不一致，请检查文件。")

    # 检查 y 是否真的是 0/1
    uniq_vals = pd.unique(pd.Series(y.reshape(-1)))
    uniq_vals = set([int(v) for v in uniq_vals if pd.notna(v)])
    if not uniq_vals.issubset({0, 1}):
        print(f"[WARN] y 中存在非 0/1 值（已尽量二值化），uniq={sorted(list(uniq_vals))[:10]} ...")

    # 提示一下 SMILES 列类型（不影响训练）
    if df[smiles_col].dtype != object:
        print(f"[WARN] 第一列({smiles_col}) 不是文本类型，但仍按你的规则将其视为 SMILES 列并忽略。")

    return X, y, feature_cols, label_cols, smiles_col, df

# =========================
# 3) 指标（macro）
# =========================
def multilabel_macro_metrics(y_true, y_prob, thresh=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= thresh).astype(int)

    L = y_true.shape[1]
    accs, precs, recs, specs = [], [], [], []
    aurocs, auprcs = [], []

    for k in range(L):
        yt = y_true[:, k]
        yp = y_pred[:, k]
        ys = y_prob[:, k]

        tp = int(np.sum((yt == 1) & (yp == 1)))
        tn = int(np.sum((yt == 0) & (yp == 0)))
        fp = int(np.sum((yt == 0) & (yp == 1)))
        fn = int(np.sum((yt == 1) & (yp == 0)))
        n = len(yt)

        accs.append((tp + tn) / n if n else np.nan)
        precs.append(tp / (tp + fp) if (tp + fp) else np.nan)
        recs.append(tp / (tp + fn) if (tp + fn) else np.nan)
        specs.append(tn / (tn + fp) if (tn + fp) else np.nan)

        if len(np.unique(yt)) == 2:
            aurocs.append(roc_auc_score(yt, ys))
            auprcs.append(average_precision_score(yt, ys))
        else:
            aurocs.append(np.nan)
            auprcs.append(np.nan)

    def nanmean(x):
        return float(np.nanmean(np.asarray(x, dtype=float)))

    return {
        "Accuracy_macro": nanmean(accs),
        "Precision_macro": nanmean(precs),
        "Recall_macro": nanmean(recs),
        "Specificity_macro": nanmean(specs),
        "AUROC_macro": nanmean(aurocs),
        "AUPRC_macro": nanmean(auprcs),
        "AUROC_valid_labels": int(np.sum(~np.isnan(aurocs))),
        "AUPRC_valid_labels": int(np.sum(~np.isnan(auprcs))),
        "n_labels": int(L),
    }

def extract_positive_proba(p, n_labels: int):
    if isinstance(p, list):
        return np.vstack([pi[:, 1] for pi in p]).T.astype(np.float32)

    p = np.asarray(p)
    if p.ndim == 3 and p.shape[-1] == 2:
        return p[:, :, 1].astype(np.float32)
    if p.ndim == 2:
        if p.shape[1] == 2 and n_labels == 1:
            return p[:, 1:2].astype(np.float32)
        return p.astype(np.float32)
    raise ValueError(f"无法解析 predict_proba 输出形状: {p.shape}")

# =========================
# 4) 近似分层 5 折：用 label cardinality 分层
# =========================
def make_stratify_target(y: np.ndarray, n_bins: int = 10):
    card = y.sum(axis=1).astype(int)
    uniq = np.unique(card)
    if len(uniq) <= 15:
        return card
    r = pd.Series(card).rank(method="average").values
    bins = pd.qcut(r, q=min(n_bins, len(np.unique(r))), labels=False, duplicates="drop")
    return np.asarray(bins, dtype=int)

def build_folds(X, y, n_splits=5, seed=42):
    strat_y = make_stratify_target(y)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return list(skf.split(X, strat_y))

# =========================
# 5) 5 折 CV 评估
# =========================
def cv_eval_one_paramset(X, y, folds, params, thresh=0.5):
    n_labels = y.shape[1]
    fold_metrics = []

    for fold_id, (tr_idx, va_idx) in enumerate(folds, start=1):
        X_tr, X_va = X[tr_idx], X[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        clf = xgb.XGBClassifier(**params)
        clf.fit(X_tr, y_tr)

        p = clf.predict_proba(X_va)
        y_prob = extract_positive_proba(p, n_labels=n_labels)

        m = multilabel_macro_metrics(y_va, y_prob, thresh=thresh)
        fold_metrics.append(m)

    keys = list(fold_metrics[0].keys())
    mean_metrics = {}
    for k in keys:
        vals = [fm[k] for fm in fold_metrics]
        if isinstance(vals[0], float):
            mean_metrics[k] = float(np.nanmean(vals))
        else:
            mean_metrics[k] = vals[-1]
    return mean_metrics

# =========================
# 6) Optuna 超参空间
# =========================
def build_trial_params(trial: optuna.Trial):
    params = dict(BASE_XGB_PARAMS)
    params.update({
        "n_estimators": trial.suggest_int("n_estimators", 300, 1200),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_float("min_child_weight", 0.5, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 20.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 5.0, log=True),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
    })
    return params

# =========================
# 7) 主流程：读特征 -> 固定folds -> Optuna(30) -> 输出+保存
# =========================
def main():
    check_xgb_version()

    feature_file = find_feature_file(FEATURE_FILE_GLOB)
    print("[INFO] Using feature file:", feature_file)

    # ✅ 按你要求的列位置切分
    X, y, feat_cols, label_cols, smiles_col, df_all = load_Xy_by_fixed_columns(
        feature_file, n_labels=138
    )
    print(f"[INFO] SMILES col = {smiles_col}")
    print(f"[INFO] y cols: {len(label_cols)} (from col2..col139)")
    print(f"[INFO] X cols: {len(feat_cols)} (from col140..end)")
    print(f"[INFO] X shape={X.shape} | y shape={y.shape}")

    folds = build_folds(X, y, n_splits=N_SPLITS, seed=RANDOM_SEED)
    print(f"[INFO] Prepared fixed {N_SPLITS}-fold splits for all trials.")

    def objective(trial: optuna.Trial):
        params = build_trial_params(trial)
        mean_metrics = cv_eval_one_paramset(X, y, folds, params, thresh=THRESH)

        score = mean_metrics["AUPRC_macro"]  # PRAUC
        print(
            f"\n[TRIAL {trial.number:02d}] score(AUPRC_macro)={score:.6f} | "
            f"AUROC={mean_metrics['AUROC_macro']:.6f} | "
            f"Acc={mean_metrics['Accuracy_macro']:.6f} | "
            f"P={mean_metrics['Precision_macro']:.6f} | "
            f"R={mean_metrics['Recall_macro']:.6f} | "
            f"Spec={mean_metrics['Specificity_macro']:.6f}"
        )

        for k, v in mean_metrics.items():
            trial.set_user_attr(k, v)

        return score

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED),
    )
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

    print("\n================ BEST =================")
    print("Best trial:", study.best_trial.number)
    print("Best AUPRC_macro:", study.best_value)
    print("Best params:")
    print(json.dumps(study.best_params, indent=2, ensure_ascii=False))

    rows = []
    for t in study.trials:
        if t.value is None:
            continue
        row = {"trial": t.number, "AUPRC_macro": t.value, **t.params}
        row.update({k: v for k, v in t.user_attrs.items() if k.endswith("_macro")})
        rows.append(row)

    res_df = pd.DataFrame(rows).sort_values("AUPRC_macro", ascending=False)
    out_csv = "./optuna_singlemodel_FIXEDCOLS_30trials_5fold_results.csv"
    res_df.to_csv(out_csv, index=False, encoding="utf-8-sig")
    print("[SAVED]", out_csv)

if __name__ == "__main__":
    main()


[INFO] Using feature file: ./Malodors_Rule&FG&StructKG_features.xlsx


[I 2026-01-30 21:53:28,575] A new study created in memory with name: no-name-8da9b755-5729-4148-b3b7-8f2b8a548595


[INFO] SMILES col = Canonical_SMILES
[INFO] y cols: 138 (from col2..col139)
[INFO] X cols: 579 (from col140..end)
[INFO] X shape=(4952, 579) | y shape=(4952, 138)
[INFO] Prepared fixed 5-fold splits for all trials.


  0%|          | 0/30 [00:00<?, ?it/s]


[TRIAL 00] score(AUPRC_macro)=0.287743 | AUROC=0.861325 | Acc=0.971347 | P=0.437249 | R=0.191157 | Spec=0.991047
[I 2026-01-31 01:29:49,162] Trial 0 finished with value: 0.2877434452439999 and parameters: {'n_estimators': 637, 'max_depth': 10, 'learning_rate': 0.08960785365368121, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.6624074561769746, 'min_child_weight': 0.7978542347074177, 'reg_lambda': 0.0017775399007348214, 'reg_alpha': 0.3426417745118369, 'gamma': 3.005575058716044}. Best is trial 0 with value: 0.2877434452439999.

[TRIAL 01] score(AUPRC_macro)=0.285543 | AUROC=0.863102 | Acc=0.971638 | P=0.424846 | R=0.187128 | Spec=0.991643
[I 2026-01-31 02:08:31,587] Trial 1 finished with value: 0.2855433860757967 and parameters: {'n_estimators': 937, 'max_depth': 3, 'learning_rate': 0.18276027831785724, 'subsample': 0.9329770563201687, 'colsample_bytree': 0.6849356442713105, 'min_child_weight': 0.8620446097910766, 'reg_lambda': 0.006149337057087106, 'reg_alpha': 4.43194278915

In [1]:
# -*- coding: utf-8 -*-
import os
import re
import glob
import json
import warnings
import numpy as np
import pandas as pd

import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.multioutput import MultiOutputClassifier

# =========================
# 0) 全局：尽量屏蔽警告 + LightGBM 日志
# =========================
os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

try:
    import lightgbm as lgb
except Exception as e:
    raise RuntimeError(
        "未检测到 lightgbm。请先安装：pip install lightgbm\n"
        f"原始错误：{repr(e)}"
    )

# =========================
# 1) 配置
# =========================
RANDOM_SEED = 42
N_TRIALS = 30
N_SPLITS = 5
THRESH = 0.5

# ✅ 改成你 XGB 那种文件（包含：SMILES + 138标签 + 特征）
FEATURE_FILE_GLOB = "./Malodors_Rule&FG&StructKG_features.xlsx"

# 固定参数（Optuna 只搜索部分超参）
BASE_LGB_PARAMS = dict(
    objective="binary",
    boosting_type="gbdt",
    n_jobs=-1,
    random_state=RANDOM_SEED,
    verbosity=-1,  # 关闭 LightGBM 日志（重要）
)

# =========================
# 2) 找文件 + 按列位置强制切分 X / y（完全照 XGB 那套）
# =========================
def find_feature_file(pattern: str):
    cands = sorted(glob.glob(pattern))
    if not cands:
        raise FileNotFoundError(
            f"找不到特征文件：{pattern}\n"
            f"请确认你保存的 xlsx 文件名，或修改 FEATURE_FILE_GLOB。"
        )
    return cands[-1]

def _to_binary01_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    将 y 的 138 列尽量转成 0/1：
    - bool -> 0/1
    - 数值 -> (>=0.5) 视为 1，否则 0
    - 字符串 -> 识别 {'1','0','true','false','yes','no'} 等
    """
    out = df.copy()

    for c in out.columns:
        s = out[c]

        if s.dtype == bool:
            out[c] = s.astype(int)
            continue

        if np.issubdtype(s.dtype, np.number):
            out[c] = (s.fillna(0).astype(float) >= 0.5).astype(int)
            continue

        ss = s.astype(str).str.strip().str.lower()
        true_set = {"1", "true", "t", "yes", "y"}
        false_set = {"0", "false", "f", "no", "n", "nan", "none", ""}

        out[c] = ss.map(lambda x: 1 if x in true_set else (0 if x in false_set else np.nan))
        if out[c].isna().any():
            coerced = pd.to_numeric(s, errors="coerce")
            if coerced.notna().any():
                out[c] = (coerced.fillna(0).astype(float) >= 0.5).astype(int)
            else:
                out[c] = out[c].fillna(0).astype(int)

    return out

def load_Xy_by_fixed_columns(xlsx_path: str, n_labels: int = 138):
    """
    固定切分（与你 XGB 脚本一致）：
      第 1 列(索引0)           -> SMILES（不进模型）
      第 2~第139列(索引1~138) -> y（共 138 列）
      第 140列起(索引>=139)   -> X（训练特征）
    """
    df = pd.read_excel(xlsx_path)

    if df.shape[1] < 1 + n_labels + 1:
        raise ValueError(
            f"列数不够：当前 {df.shape[1]} 列，但至少需要 1(SMILES)+{n_labels}(y)+1(X) = {1+n_labels+1} 列。\n"
            f"请检查文件是否缺列/被截断。"
        )

    smiles_col = df.columns[0]
    label_cols = list(df.columns[1 : 1 + n_labels])
    feature_cols = list(df.columns[1 + n_labels : ])

    # y：强制二值化
    y_df = _to_binary01_df(df[label_cols])

    # X：数值化 + NaN填0
    X_df = df[feature_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).astype(np.float32)

    X = X_df.values
    y = y_df.fillna(0).astype(int).values

    if X.shape[0] != y.shape[0]:
        raise ValueError("X 和 y 行数不一致，请检查文件。")

    uniq_vals = pd.unique(pd.Series(y.reshape(-1)))
    uniq_vals = set([int(v) for v in uniq_vals if pd.notna(v)])
    if not uniq_vals.issubset({0, 1}):
        print(f"[WARN] y 中存在非 0/1 值（已尽量二值化），uniq={sorted(list(uniq_vals))[:10]} ...")

    if df[smiles_col].dtype != object:
        print(f"[WARN] 第一列({smiles_col}) 不是文本类型，但仍按规则将其视为 SMILES 列并忽略。")

    return X, y, feature_cols, label_cols, smiles_col, df

# =========================
# 3) 指标（macro）
# =========================
def multilabel_macro_metrics(y_true, y_prob, thresh=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= thresh).astype(int)

    L = y_true.shape[1]
    accs, precs, recs, specs = [], [], [], []
    aurocs, auprcs = [], []

    for k in range(L):
        yt = y_true[:, k]
        yp = y_pred[:, k]
        ys = y_prob[:, k]

        tp = int(np.sum((yt == 1) & (yp == 1)))
        tn = int(np.sum((yt == 0) & (yp == 0)))
        fp = int(np.sum((yt == 0) & (yp == 1)))
        fn = int(np.sum((yt == 1) & (yp == 0)))
        n = len(yt)

        accs.append((tp + tn) / n if n else np.nan)
        precs.append(tp / (tp + fp) if (tp + fp) else np.nan)
        recs.append(tp / (tp + fn) if (tp + fn) else np.nan)
        specs.append(tn / (tn + fp) if (tn + fp) else np.nan)

        if len(np.unique(yt)) == 2:
            aurocs.append(roc_auc_score(yt, ys))
            auprcs.append(average_precision_score(yt, ys))
        else:
            aurocs.append(np.nan)
            auprcs.append(np.nan)

    def nanmean(x):
        return float(np.nanmean(np.asarray(x, dtype=float)))

    return {
        "Accuracy_macro": nanmean(accs),
        "Precision_macro": nanmean(precs),
        "Recall_macro": nanmean(recs),
        "Specificity_macro": nanmean(specs),
        "AUROC_macro": nanmean(aurocs),
        "AUPRC_macro": nanmean(auprcs),
        "AUROC_valid_labels": int(np.sum(~np.isnan(aurocs))),
        "AUPRC_valid_labels": int(np.sum(~np.isnan(auprcs))),
        "n_labels": int(L),
    }

def extract_positive_proba(p, n_labels: int):
    # MultiOutputClassifier.predict_proba -> list，长度=labels，每个是 (n,2) 或 (n,1)
    if isinstance(p, list):
        out = np.zeros((p[0].shape[0], n_labels), dtype=np.float32)
        for k in range(n_labels):
            pk = p[k]
            if pk.ndim != 2:
                raise ValueError(f"predict_proba[{k}] 形状异常: {pk.shape}")
            if pk.shape[1] == 2:
                out[:, k] = pk[:, 1].astype(np.float32)
            elif pk.shape[1] == 1:
                # 单类：全 0 或全 1；sklearn 有时只给一列
                out[:, k] = 0.0
            else:
                raise ValueError(f"predict_proba[{k}] 类别数异常: {pk.shape[1]}")
        return out

    p = np.asarray(p)
    if p.ndim == 3 and p.shape[-1] == 2:
        return p[:, :, 1].astype(np.float32)
    if p.ndim == 2:
        if p.shape[1] == 2 and n_labels == 1:
            return p[:, 1:2].astype(np.float32)
        return p.astype(np.float32)
    raise ValueError(f"无法解析 predict_proba 输出形状: {p.shape}")

# =========================
# 4) 近似分层 5 折：用 label cardinality 分层
# =========================
def make_stratify_target(y: np.ndarray, n_bins: int = 10):
    card = y.sum(axis=1).astype(int)
    uniq = np.unique(card)
    if len(uniq) <= 15:
        return card
    r = pd.Series(card).rank(method="average").values
    bins = pd.qcut(r, q=min(n_bins, len(np.unique(r))), labels=False, duplicates="drop")
    return np.asarray(bins, dtype=int)

def build_folds(X, y, n_splits=5, seed=42):
    strat_y = make_stratify_target(y)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return list(skf.split(X, strat_y))

# =========================
# 5) 5 折 CV 评估（LightGBM + MultiOutput）
# =========================
def cv_eval_one_paramset(X, y, folds, params, thresh=0.5):
    n_labels = y.shape[1]
    fold_metrics = []

    for fold_id, (tr_idx, va_idx) in enumerate(folds, start=1):
        X_tr, X_va = X[tr_idx], X[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        base_est = lgb.LGBMClassifier(**params)
        clf = MultiOutputClassifier(base_est, n_jobs=1)  # 外层不并行，避免线程打架
        clf.fit(X_tr, y_tr)

        p = clf.predict_proba(X_va)
        y_prob = extract_positive_proba(p, n_labels=n_labels)

        m = multilabel_macro_metrics(y_va, y_prob, thresh=thresh)
        fold_metrics.append(m)

    keys = list(fold_metrics[0].keys())
    mean_metrics = {}
    for k in keys:
        vals = [fm[k] for fm in fold_metrics]
        if isinstance(vals[0], float):
            mean_metrics[k] = float(np.nanmean(vals))
        else:
            mean_metrics[k] = vals[-1]
    return mean_metrics

# =========================
# 6) Optuna 超参空间（LightGBM）
# =========================
def build_trial_params(trial: optuna.Trial):
    params = dict(BASE_LGB_PARAMS)
    params.update({
        "n_estimators": trial.suggest_int("n_estimators", 300, 2000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),

        "num_leaves": trial.suggest_int("num_leaves", 31, 255),
        "max_depth": trial.suggest_int("max_depth", -1, 20),  # -1 表示不限制
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 80),

        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),

        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 50.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),

        "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 5.0),
    })
    return params

# =========================
# 7) 主流程：读特征 -> 固定folds -> Optuna -> 输出
# =========================
def main():
    feature_file = find_feature_file(FEATURE_FILE_GLOB)
    print("[INFO] Using feature file:", feature_file)

    # ✅ 按 XGB 同款固定列切分
    X, y, feat_cols, label_cols, smiles_col, df_all = load_Xy_by_fixed_columns(
        feature_file, n_labels=138
    )
    print(f"[INFO] SMILES col = {smiles_col}")
    print(f"[INFO] y cols: {len(label_cols)} (from col2..col139)")
    print(f"[INFO] X cols: {len(feat_cols)} (from col140..end)")
    print(f"[INFO] X shape={X.shape} | y shape={y.shape}")

    folds = build_folds(X, y, n_splits=N_SPLITS, seed=RANDOM_SEED)
    print(f"[INFO] Prepared fixed {N_SPLITS}-fold splits for all trials.")

    def objective(trial: optuna.Trial):
        params = build_trial_params(trial)
        mean_metrics = cv_eval_one_paramset(X, y, folds, params, thresh=THRESH)

        score = mean_metrics["AUPRC_macro"]
        print(
            f"\n[TRIAL {trial.number:02d}] score(AUPRC_macro)={score:.6f} | "
            f"AUROC={mean_metrics['AUROC_macro']:.6f} | "
            f"Acc={mean_metrics['Accuracy_macro']:.6f} | "
            f"P={mean_metrics['Precision_macro']:.6f} | "
            f"R={mean_metrics['Recall_macro']:.6f} | "
            f"Spec={mean_metrics['Specificity_macro']:.6f}"
        )

        for k, v in mean_metrics.items():
            trial.set_user_attr(k, v)

        return score

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED),
    )
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

    print("\n================ BEST =================")
    print("Best trial:", study.best_trial.number)
    print("Best AUPRC_macro:", study.best_value)
    print("Best params:")
    print(json.dumps(study.best_params, indent=2, ensure_ascii=False))

    rows = []
    for t in study.trials:
        if t.value is None:
            continue
        row = {"trial": t.number, "AUPRC_macro": t.value, **t.params}
        row.update({k: v for k, v in t.user_attrs.items() if k.endswith("_macro")})
        rows.append(row)

    res_df = pd.DataFrame(rows).sort_values("AUPRC_macro", ascending=False)
    out_csv = "./optuna_singlemodel_LIGHTGBM_FIXEDCOLS_30trials_5fold_results.csv"
    res_df.to_csv(out_csv, index=False, encoding="utf-8-sig")
    print("[SAVED]", out_csv)

if __name__ == "__main__":
    main()


[INFO] Using feature file: ./Malodors_Rule&FG&StructKG_features.xlsx


[I 2026-02-10 10:20:35,130] A new study created in memory with name: no-name-05c02e67-5d29-44bb-b140-879901f3315e


[INFO] SMILES col = Canonical_SMILES
[INFO] y cols: 138 (from col2..col139)
[INFO] X cols: 579 (from col140..end)
[INFO] X shape=(4952, 579) | y shape=(4952, 138)
[INFO] Prepared fixed 5-fold splits for all trials.


  0%|          | 0/30 [00:00<?, ?it/s]


[TRIAL 00] score(AUPRC_macro)=0.229589 | AUROC=0.844478 | Acc=0.972074 | P=0.728354 | R=0.055074 | Spec=0.996251
[I 2026-02-10 10:25:48,970] Trial 0 finished with value: 0.2295894779278432 and parameters: {'n_estimators': 937, 'learning_rate': 0.17254716573280354, 'num_leaves': 195, 'max_depth': 12, 'min_child_samples': 16, 'subsample': 0.662397808134481, 'colsample_bytree': 0.6232334448672797, 'reg_lambda': 11.752647960576219, 'reg_alpha': 0.002570603566117598, 'min_split_gain': 3.540362888980227}. Best is trial 0 with value: 0.2295894779278432.

[TRIAL 01] score(AUPRC_macro)=0.271068 | AUROC=0.853048 | Acc=0.972302 | P=0.539744 | R=0.112264 | Spec=0.994865
[I 2026-02-10 10:28:06,274] Trial 1 finished with value: 0.2710679345156494 and parameters: {'n_estimators': 335, 'learning_rate': 0.18276027831785724, 'num_leaves': 218, 'max_depth': 3, 'min_child_samples': 18, 'subsample': 0.6733618039413735, 'colsample_bytree': 0.7216968971838151, 'reg_lambda': 0.2922905212920093, 'reg_alpha': 

In [2]:
# -*- coding: utf-8 -*-
import os
import re
import glob
import json
import warnings
import numpy as np
import pandas as pd

import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.ensemble import RandomForestClassifier

# =========================
# 0) 配置 & 尽量屏蔽警告
# =========================
RANDOM_SEED = 42
N_TRIALS = 30
N_SPLITS = 5
THRESH = 0.5

os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ✅ 改成“固定列切分”那种文件（SMILES + 138标签 + 特征）
FEATURE_FILE_GLOB = "./Malodors_Rule&FG&StructKG_features.xlsx"
# 如果你确实就是 Morgan-only 且文件结构也满足“SMILES+138标签+Morgan特征”，也可以改成：
# FEATURE_FILE_GLOB = "./Malodors_transformed_MORGAN_features*.xlsx"

# RandomForest 固定参数（Optuna 只搜索部分超参）
BASE_RF_PARAMS = dict(
    n_jobs=-1,
    random_state=RANDOM_SEED,
)

# =========================
# 1) 找文件
# =========================
def find_feature_file(pattern: str):
    cands = sorted(glob.glob(pattern))
    if not cands:
        raise FileNotFoundError(
            f"找不到特征文件：{pattern}\n"
            f"请确认你保存的 xlsx 文件名，或修改 FEATURE_FILE_GLOB。"
        )
    return cands[-1]

# =========================
# 2) 固定列位置切分 X / y（与你 XGB 脚本完全一致）
# =========================
def _to_binary01_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    将 y 的 138 列尽量转成 0/1：
    - bool -> 0/1
    - 数值 -> (>=0.5) 视为 1，否则 0
    - 字符串 -> 识别 {'1','0','true','false','yes','no'} 等
    """
    out = df.copy()

    for c in out.columns:
        s = out[c]

        if s.dtype == bool:
            out[c] = s.astype(int)
            continue

        if np.issubdtype(s.dtype, np.number):
            out[c] = (s.fillna(0).astype(float) >= 0.5).astype(int)
            continue

        ss = s.astype(str).str.strip().str.lower()
        true_set = {"1", "true", "t", "yes", "y"}
        false_set = {"0", "false", "f", "no", "n", "nan", "none", ""}

        out[c] = ss.map(lambda x: 1 if x in true_set else (0 if x in false_set else np.nan))
        if out[c].isna().any():
            coerced = pd.to_numeric(s, errors="coerce")
            if coerced.notna().any():
                out[c] = (coerced.fillna(0).astype(float) >= 0.5).astype(int)
            else:
                out[c] = out[c].fillna(0).astype(int)

    return out

def load_Xy_by_fixed_columns(xlsx_path: str, n_labels: int = 138):
    """
    固定切分规则：
      第 1 列(索引0)           -> SMILES（不进模型）
      第 2~第139列(索引1~138) -> y（共 138 列）
      第 140列起(索引>=139)   -> X（训练特征）
    """
    df = pd.read_excel(xlsx_path)

    if df.shape[1] < 1 + n_labels + 1:
        raise ValueError(
            f"列数不够：当前 {df.shape[1]} 列，但至少需要 1(SMILES)+{n_labels}(y)+1(X) = {1+n_labels+1} 列。\n"
            f"请检查文件是否缺列/被截断。"
        )

    smiles_col = df.columns[0]
    label_cols = list(df.columns[1 : 1 + n_labels])
    feature_cols = list(df.columns[1 + n_labels : ])

    # y：强制二值化
    y_df = _to_binary01_df(df[label_cols])

    # X：数值化 + NaN填0
    X_df = df[feature_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).astype(np.float32)

    X = X_df.values
    y = y_df.fillna(0).astype(int).values

    if X.shape[0] != y.shape[0]:
        raise ValueError("X 和 y 行数不一致，请检查文件。")

    uniq_vals = pd.unique(pd.Series(y.reshape(-1)))
    uniq_vals = set([int(v) for v in uniq_vals if pd.notna(v)])
    if not uniq_vals.issubset({0, 1}):
        print(f"[WARN] y 中存在非 0/1 值（已尽量二值化），uniq={sorted(list(uniq_vals))[:10]} ...")

    if df[smiles_col].dtype != object:
        print(f"[WARN] 第一列({smiles_col}) 不是文本类型，但仍按规则将其视为 SMILES 列并忽略。")

    return X, y, feature_cols, label_cols, smiles_col, df

# =========================
# 3) 指标（macro）
# =========================
def multilabel_macro_metrics(y_true, y_prob, thresh=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= thresh).astype(int)

    L = y_true.shape[1]
    accs, precs, recs, specs = [], [], [], []
    aurocs, auprcs = [], []

    for k in range(L):
        yt = y_true[:, k]
        yp = y_pred[:, k]
        ys = y_prob[:, k]

        tp = int(np.sum((yt == 1) & (yp == 1)))
        tn = int(np.sum((yt == 0) & (yp == 0)))
        fp = int(np.sum((yt == 0) & (yp == 1)))
        fn = int(np.sum((yt == 1) & (yp == 0)))
        n = len(yt)

        accs.append((tp + tn) / n if n else np.nan)
        precs.append(tp / (tp + fp) if (tp + fp) else np.nan)
        recs.append(tp / (tp + fn) if (tp + fn) else np.nan)
        specs.append(tn / (tn + fp) if (tn + fp) else np.nan)

        if len(np.unique(yt)) == 2:
            aurocs.append(roc_auc_score(yt, ys))
            auprcs.append(average_precision_score(yt, ys))
        else:
            aurocs.append(np.nan)
            auprcs.append(np.nan)

    def nanmean(x):
        return float(np.nanmean(np.asarray(x, dtype=float)))

    return {
        "Accuracy_macro": nanmean(accs),
        "Precision_macro": nanmean(precs),
        "Recall_macro": nanmean(recs),
        "Specificity_macro": nanmean(specs),
        "AUROC_macro": nanmean(aurocs),
        "AUPRC_macro": nanmean(auprcs),
        "AUROC_valid_labels": int(np.sum(~np.isnan(aurocs))),
        "AUPRC_valid_labels": int(np.sum(~np.isnan(auprcs))),
        "n_labels": int(L),
    }

def extract_positive_proba(p, n_labels: int, classes_list=None):
    """
    multi-output RF 的 predict_proba: list，长度=labels，每个 (n, n_classes_k)
    处理某标签在该 fold 训练集里只有单类 => (n,1)
    """
    if isinstance(p, list):
        out = np.zeros((p[0].shape[0], n_labels), dtype=np.float32)
        for k in range(n_labels):
            pk = p[k]
            if pk.ndim != 2:
                raise ValueError(f"predict_proba[{k}] 形状异常: {pk.shape}")

            if pk.shape[1] == 2:
                if classes_list is not None and len(classes_list) == n_labels:
                    cls = list(classes_list[k])
                    if 1 in cls:
                        j = cls.index(1)
                        out[:, k] = pk[:, j].astype(np.float32)
                    else:
                        out[:, k] = 0.0
                else:
                    out[:, k] = pk[:, 1].astype(np.float32)

            elif pk.shape[1] == 1:
                if classes_list is not None and len(classes_list) == n_labels:
                    only_cls = int(list(classes_list[k])[0])
                    out[:, k] = 1.0 if only_cls == 1 else 0.0
                else:
                    out[:, k] = 0.0
            else:
                raise ValueError(f"predict_proba[{k}] 类别数异常: {pk.shape[1]}")
        return out

    p = np.asarray(p)
    if p.ndim == 3 and p.shape[-1] == 2:
        return p[:, :, 1].astype(np.float32)
    if p.ndim == 2:
        if p.shape[1] == 2 and n_labels == 1:
            return p[:, 1:2].astype(np.float32)
        return p.astype(np.float32)
    raise ValueError(f"无法解析 predict_proba 输出形状: {p.shape}")

# =========================
# 4) 近似分层 5 折：用 label cardinality 分层
# =========================
def make_stratify_target(y: np.ndarray, n_bins: int = 10):
    card = y.sum(axis=1).astype(int)
    uniq = np.unique(card)
    if len(uniq) <= 15:
        return card
    r = pd.Series(card).rank(method="average").values
    bins = pd.qcut(r, q=min(n_bins, len(np.unique(r))), labels=False, duplicates="drop")
    return np.asarray(bins, dtype=int)

def build_folds(X, y, n_splits=5, seed=42):
    strat_y = make_stratify_target(y)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return list(skf.split(X, strat_y))

# =========================
# 5) 5 折 CV 评估（RF）
# =========================
def cv_eval_one_paramset(X, y, folds, params, thresh=0.5):
    n_labels = y.shape[1]
    fold_metrics = []

    for fold_id, (tr_idx, va_idx) in enumerate(folds, start=1):
        X_tr, X_va = X[tr_idx], X[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        clf = RandomForestClassifier(**params)
        clf.fit(X_tr, y_tr)

        p = clf.predict_proba(X_va)  # list(length=L)
        y_prob = extract_positive_proba(p, n_labels=n_labels, classes_list=getattr(clf, "classes_", None))

        m = multilabel_macro_metrics(y_va, y_prob, thresh=thresh)
        fold_metrics.append(m)

    keys = list(fold_metrics[0].keys())
    mean_metrics = {}
    for k in keys:
        vals = [fm[k] for fm in fold_metrics]
        if isinstance(vals[0], float):
            mean_metrics[k] = float(np.nanmean(vals))
        else:
            mean_metrics[k] = vals[-1]
    return mean_metrics

# =========================
# 6) Optuna 超参空间（RF）
# =========================
def build_trial_params(trial: optuna.Trial):
    params = dict(BASE_RF_PARAMS)

    max_depth_raw = trial.suggest_int("max_depth", 0, 40)  # 0 表示 None
    max_depth = None if max_depth_raw == 0 else max_depth_raw

    params.update({
        "n_estimators": trial.suggest_int("n_estimators", 300, 1500),
        "max_depth": max_depth,
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", 0.3, 0.5, 0.8, 1.0]),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "bootstrap": trial.suggest_categorical("bootstrap", [True, False]),
        "criterion": trial.suggest_categorical("criterion", ["gini", "entropy", "log_loss"]),
    })
    return params

# =========================
# 7) 主流程：读特征 -> 固定folds -> Optuna(30) -> 输出
# =========================
def main():
    feature_file = find_feature_file(FEATURE_FILE_GLOB)
    print("[INFO] Using feature file:", feature_file)

    # ✅ 固定列切分（同理改这里）
    X, y, feat_cols, label_cols, smiles_col, df_all = load_Xy_by_fixed_columns(
        feature_file, n_labels=138
    )
    print(f"[INFO] SMILES col = {smiles_col}")
    print(f"[INFO] y cols: {len(label_cols)} (from col2..col139)")
    print(f"[INFO] X cols: {len(feat_cols)} (from col140..end)")
    print(f"[INFO] X shape={X.shape} | y shape={y.shape}")

    folds = build_folds(X, y, n_splits=N_SPLITS, seed=RANDOM_SEED)
    print(f"[INFO] Prepared fixed {N_SPLITS}-fold splits for all trials.")

    def objective(trial: optuna.Trial):
        params = build_trial_params(trial)
        mean_metrics = cv_eval_one_paramset(X, y, folds, params, thresh=THRESH)

        score = mean_metrics["AUPRC_macro"]
        print(
            f"\n[TRIAL {trial.number:02d}] score(AUPRC_macro)={score:.6f} | "
            f"AUROC={mean_metrics['AUROC_macro']:.6f} | "
            f"Acc={mean_metrics['Accuracy_macro']:.6f} | "
            f"P={mean_metrics['Precision_macro']:.6f} | "
            f"R={mean_metrics['Recall_macro']:.6f} | "
            f"Spec={mean_metrics['Specificity_macro']:.6f}"
        )

        for k, v in mean_metrics.items():
            trial.set_user_attr(k, v)

        return score

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED),
    )
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

    print("\n================ BEST =================")
    print("Best trial:", study.best_trial.number)
    print("Best AUPRC_macro:", study.best_value)
    print("Best params:")
    print(json.dumps(study.best_params, indent=2, ensure_ascii=False))

    rows = []
    for t in study.trials:
        if t.value is None:
            continue
        row = {"trial": t.number, "AUPRC_macro": t.value, **t.params}
        row.update({k: v for k, v in t.user_attrs.items() if k.endswith("_macro")})
        rows.append(row)

    res_df = pd.DataFrame(rows).sort_values("AUPRC_macro", ascending=False)
    out_csv = "./optuna_singlemodel_RF_FIXEDCOLS_30trials_5fold_results.csv"
    res_df.to_csv(out_csv, index=False, encoding="utf-8-sig")
    print("[SAVED]", out_csv)

if __name__ == "__main__":
    main()


[INFO] Using feature file: ./Malodors_Rule&FG&StructKG_features.xlsx


[I 2026-02-10 14:24:06,454] A new study created in memory with name: no-name-9fad1309-be24-4360-bab3-6ca563a79989


[INFO] SMILES col = Canonical_SMILES
[INFO] y cols: 138 (from col2..col139)
[INFO] X cols: 579 (from col140..end)
[INFO] X shape=(4952, 579) | y shape=(4952, 138)
[INFO] Prepared fixed 5-fold splits for all trials.


  0%|          | 0/30 [00:00<?, ?it/s]


[TRIAL 00] score(AUPRC_macro)=0.211856 | AUROC=0.759812 | Acc=0.971035 | P=0.465522 | R=0.116202 | Spec=0.992313
[I 2026-02-10 15:06:42,602] Trial 0 finished with value: 0.2118556991866226 and parameters: {'max_depth': 15, 'n_estimators': 1441, 'max_features': 1.0, 'min_samples_split': 13, 'min_samples_leaf': 8, 'bootstrap': False, 'criterion': 'gini'}. Best is trial 0 with value: 0.2118556991866226.

[TRIAL 01] score(AUPRC_macro)=0.271890 | AUROC=0.865927 | Acc=0.971815 | P=0.746178 | R=0.048064 | Spec=0.997195
[I 2026-02-10 15:16:45,567] Trial 1 finished with value: 0.2718899056703045 and parameters: {'max_depth': 7, 'n_estimators': 665, 'max_features': 0.5, 'min_samples_split': 8, 'min_samples_leaf': 5, 'bootstrap': True, 'criterion': 'entropy'}. Best is trial 1 with value: 0.2718899056703045.

[TRIAL 02] score(AUPRC_macro)=0.288998 | AUROC=0.850348 | Acc=0.972765 | P=0.634667 | R=0.094300 | Spec=0.995310
[I 2026-02-10 15:24:39,769] Trial 2 finished with value: 0.28899763912549353 